In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd



In [2]:
gdf = gpd.read_file('./data/blocks_investment.geojson')
gdf.head()

,land_use,land_area,land_value_before,built_area,land_value,construction_cost,investment_need,NPV,IRR,ROI,PP_years,EI,spatial_potential,INV,geometry
0,LandUse.AGRICULTURE,202399.73,1.040179e+08,12550.41,3.132176e+08,3.137603e+08,6.269778e+08,-3.417592e+08,0.06,2.12,NaN,0.00,3.0,31.07,"POLYGON ((388511.373 6644951.946, 388558.037 6..."
1,LandUse.BUSINESS,201529.49,1.035707e+08,40777.54,3.488025e+08,2.242764e+09,2.591567e+09,5.066266e+08,0.22,10.07,11.23,65.99,2.0,59.36,"POLYGON ((388558.037 6645102.374, 388511.373 6..."
2,LandUse.TRANSPORT,164663.86,8.462455e+07,24985.65,2.232646e+08,4.497417e+08,6.730063e+08,-6.067684e+07,0.16,6.49,NaN,0.00,4.0,41.43,"POLYGON ((387957.434 6643444.937, 388007.492 6..."
3,LandUse.RESIDENTIAL,161571.83,8.303549e+07,72568.69,7.195687e+08,3.265591e+09,3.985160e+09,1.583141e+08,0.19,3.92,7.76,38.36,0.0,22.47,"POLYGON ((388007.492 6643608.409, 387957.434 6..."
4,None,NaN,3.494860e+03,NaN,5.723364e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((388001.306 6643195.969, 388000.75 66..."


In [8]:
from urbanomy.methods.socio_economic_indicators.constants import (
    DEFAULT_VA_PER_M2_OPS,
    DEFAULT_JOBS_PER_M2,
    DEFAULT_WAGE_BY_USE,
    DEFAULT_PROFIT_SHARE_OPS,
    DEFAULT_CAPEX_CAPITALIZABLE_SHARE,
    DEFAULT_AMORTIZATION_RATES,
)

project_cfg = {
    "population": 600_000,  # Население базовой территорииА
    "employment_share": 0.64,  # Доля занятых в базе (по умолчанию 0.62)
    "avg_wage_base": 74_000,  # Средняя зарплата до проекта
    "build_wage_share": 0.27,  # Доля зарплаты в инвестициях на стройке
    "build_profit_margin": 0.065,  # Рентабельность строительства
    "tax_rates": {  # Налоговые ставки проекта
        "pit": 0.12,  # НДФЛ
        "cit": 0.17,  # налог на прибыль
        "prop": 0.025,  # налог на имущество
        "land": 0.015,  # земельный налог
    },
    "va_coeff_build": {  # Коэффициенты валовой добавленной стоимости на стройке
        "default": 0.50,  # базовый коэффициент для прочих категорий
        "business": 0.55,  # повышаем мультипликатор для деловой застройки
        "special": 0.52,  
    },
    "va_per_m2_ops": {  # ВДС на квадратный метр в эксплуатации
        **DEFAULT_VA_PER_M2_OPS,
        "business": 13_500.0,  
        "transport": 5_500.0,  
    },
    "jobs_per_m2": {  # Рабочие места на квадратный метр
        **DEFAULT_JOBS_PER_M2,
        "business": 1 / 20, 
        "special": 1 / 28, 
    },
    "wage_by_use": {  # Средние зарплаты по видам использования земли
        **DEFAULT_WAGE_BY_USE,
        "business": 92_000, 
        "transport": 63_000,  
    },
    "profit_share_ops": {  # Доли прибыли в обороте в эксплуатации
        **DEFAULT_PROFIT_SHARE_OPS,
        "business": 0.20, 
        "special": 0.16,  
    },
    "capex_capitalizable_share": {  # Доля капзатрат, переходящих в основные средства
        **DEFAULT_CAPEX_CAPITALIZABLE_SHARE,
        "transport": 0.92,  
    },
    "amortization_rates": {  # Годовые нормы амортизации
        **DEFAULT_AMORTIZATION_RATES,
        "business": 0.032,  
        "special": 0.045,  
}}


In [9]:
from urbanomy.methods.socio_economic_indicators.sei_calculate import SEREstimator

deafaut_cfg = {
    "population": 600_000,
}

est = SEREstimator(deafaut_cfg) # or use project_cfg
result = est.compute(gdf, pretty=True)
result

,indicator,delta_total
0,Объём инвестиций в основной капитал на душу на...,90 278
1,Валовый региональный продукт на душу населения,53 874
2,Доходы бюджета территории,25 766 005 578
3,Средний уровень заработной платы,492
4,Износ основного фонда (тыс. руб.),1 749 084
